# GNN-C notebook trainer

This notebook is a GPU worker only. The local API stays the source of truth for CAD, consent, compile, and the model registry.

1. Run the local API.
2. Expose it with a public HTTPS tunnel (`cloudflared tunnel --url http://127.0.0.1:8000` or `ngrok http 8000`). Colab and Kaggle cannot reach `localhost`.
3. Paste the public origin into `BASE_URL` (no trailing slash).
4. Train here, then download `face_gnn.pt` and `metrics.json`.
5. Register locally as a **candidate** (never auto-promoted):

```bash
curl -F checkpoint=@face_gnn.pt -F metrics=@metrics.json \
  http://127.0.0.1:8000/api/v1/admin/models
```

Promotion still uses the existing local `promote` CLI. Set `FORCE = True` only for experiments that ignore the 50-file / 2-class gate.

In [ ]:
BASE_URL = "https://YOUR_TUNNEL"
FORCE = False

In [ ]:
%pip install -q torch numpy requests

In [ ]:
from __future__ import annotations

import importlib.util
import json
from pathlib import Path

import requests

base = BASE_URL.rstrip("/")
kernel_path = Path("face_train_kernel.py")
kernel_resp = requests.get(f"{base}/api/v1/admin/datasets/face/kernel.py", timeout=120)
kernel_resp.raise_for_status()
kernel_path.write_text(kernel_resp.text, encoding="utf-8")

spec = importlib.util.spec_from_file_location("face_train_kernel", kernel_path)
kernel = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(kernel)

bundle_resp = requests.get(f"{base}/api/v1/admin/datasets/face", timeout=300)
bundle_resp.raise_for_status()
payload = bundle_resp.json()
bundle = payload.get("data", payload)
eligibility = bundle.get("eligibility") or {}
print("labeled_jobs", eligibility.get("labeled_jobs"))
print("n_classes", eligibility.get("n_classes"))
print("ok", eligibility.get("ok"))
print("reasons", eligibility.get("reasons"))
print("graphs", len(bundle.get("graphs") or []))

In [ ]:
checkpoint_bytes, metrics = kernel.run_training_from_bundle(bundle, force=FORCE)
per_class = metrics.get("per_class_f1") or {}
print("held-out macro_f1", metrics.get("macro_f1"))
print("per-class F1", per_class)
print("confusion", metrics.get("confusion_matrix"))
print("heuristic_macro_f1", metrics.get("heuristic_macro_f1"))
print("room_count_error", metrics.get("room_count_error"))
print("false_room_rate", metrics.get("false_room_rate"))
print("delete_rate_proxy", metrics.get("delete_rate_proxy"))
print("room_iou_at_0_5", metrics.get("room_iou_at_0_5"))
print("held_out_passed", metrics.get("held_out_passed"))

In [ ]:
Path("face_gnn.pt").write_bytes(checkpoint_bytes)
Path("metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("wrote face_gnn.pt", len(checkpoint_bytes), "bytes")
print("wrote metrics.json")